# 09.2 — SLURM dispatch

A full experiment isn't one training run — it's *hundreds*: every architecture variant × every recording session × every cross-validation fold. You can't run those serially (it would take months) or launch each by hand. On an HPC cluster you submit them as a **SLURM array job**: one `sbatch` script that the scheduler expands into N parallel tasks, each identified by a task number. This notebook is how the project turns one sweep into a cluster-ready `.slurm` script — the `sweep-emit-slurm` command — and the single-integer indexing that makes it work.

This notebook covers:

* The bash-loop → SLURM-array-job model.
* The single integer sweep index → a config point (`lookup`).
* `SessionRunIDX`: the array task ID → (session, fold) decomposition.
* `render_slurm_template` — the emitted sbatch script.

**Prerequisites:** [09.1 environment detection](09.1_environment_detection.ipynb), [00.5 the command line](../00_orientation/00.5_the_command_line_for_matlab_users.ipynb).

## Section 1 — What MATLAB does

`SLURMPARAMETERS_cgg_runAutoEncoder_v2.m` defines the sweep, and a bash wrapper submits it. The MATLAB indexing uses two numbers — `SLURMChoice` (which parameter group) and `SLURMIDX` (which value within it):

```bash
# The MATLAB-era model: a bash script submitting an array job
sbatch --array=1-35 run_encoder.slurm    # 35 = sessions × folds for one config
# inside run_encoder.slurm, MATLAB reads $SLURM_ARRAY_TASK_ID to pick the (session, fold)
```

The port keeps this exact structure — a SLURM *array job* whose task ID selects the (session, fold) — but flattens the MATLAB `(SLURMChoice, SLURMIDX)` config coordinates into a **single integer sweep index**, and emits the `.slurm` script from Python via `sweep-emit-slurm` (mirroring the MATLAB sweep table, with `SessionRunIDX` ordering preserved).

## Section 2 — The Python concepts you need

### 2.1 — Why an array job

Say you have 147 config variants, 7 sessions, and 5 folds. That's 147 × 35 = 5,145 training runs. Three ways to run them:

* **Serially** — one after another. Months of wall-clock. No.
* **A bash loop submitting one job each** — thousands of `sbatch` calls, thousands of job IDs to track. Works, but clumsy and hammers the scheduler.
* **A SLURM array job** — *one* `sbatch` submission that the scheduler expands into N tasks, each with a distinct `$SLURM_ARRAY_TASK_ID`, run in parallel up to a throttle. One job ID, clean bookkeeping, native scheduling.

The array job is the standard HPC pattern, and it's what `sweep-emit-slurm` generates.

### 2.2 — The single integer sweep index

The config axis of the sweep is *flattened* to a single integer, `1..N`. Each index maps to one config point — a set of overrides on the base config. Look one up:

In [ ]:
from neural_data_decoding.sweeps import dispatcher

print("total config points in the sweep:", dispatcher.total_sweep_count())
for idx in (1, 5, 50):
    entry = dispatcher.lookup(idx)
    print(f"\nsweep index {idx}: {entry.description}")
    print(f"   overrides: {entry.overrides}")
    print(f"   MATLAB coords: SLURMChoice={entry.matlab_choice}, SLURMIDX={entry.matlab_idx}")

`dispatcher.lookup(index)` returns a `SweepEntry` — its `description`, the config `overrides` it applies (e.g. `{"model_name": "Feedforward", ...}`), and the original MATLAB `(SLURMChoice, SLURMIDX)` coordinates it flattened. The whole sweep is `1..147` (here), so a single integer names any config variant. Flattening two MATLAB coordinates into one index means the cluster job needs to pass only *one* number to select a config — simpler than threading two.

### 2.3 — `SessionRunIDX`: the array task ID

In [ ]:
# One config still runs across every (session, fold). That's the ARRAY axis.
num_sessions, num_folds = 7, 5
total_tasks = num_sessions * num_folds
print(f"array tasks for one config: {num_sessions} sessions × {num_folds} folds = {total_tasks}")
print()
# SLURM_ARRAY_TASK_ID (1..35) is SessionRunIDX; the CLI decomposes it into (session, fold).
# MATLAB ordering is fold-across-sessions: all sessions for fold 1, then fold 2, ...
print("SessionRunIDX → (session, fold), MATLAB fold-across-sessions ordering:")
for session_run_idx in (1, 7, 8, 35):
    fold = (session_run_idx - 1) // num_sessions + 1      # fold-major
    session = (session_run_idx - 1) % num_sessions + 1
    print(f"   SessionRunIDX {session_run_idx:2} → session {session}, fold {fold}")

So the full sweep is *two* axes: the **sweep index** (which config, `1..147`) and **`SessionRunIDX`** (which session×fold, `1..35`). A `sweep-emit-slurm` for one config emits an array job over `SessionRunIDX`; the cluster's `$SLURM_ARRAY_TASK_ID` *is* the `SessionRunIDX`, and the Python CLI decomposes it into `(session, fold)` at runtime. The ordering is **fold-across-sessions** — all sessions for fold 1 before any of fold 2 — matching MATLAB (Critical detail from the Milestone D plan: fold-1-across-all-sessions runs before fold-2, so you get one complete fold of every session early). Each task trains exactly one `(config, session, fold)` combination.

### 2.4 — The emitted sbatch script

In [ ]:
from neural_data_decoding.sweeps.slurm_template import render_slurm_template, SlurmTemplateOptions
from neural_data_decoding.sweeps.user_identity import UserIdentity

# A generic identity (real runs resolve YOUR identity; here we use a placeholder):
me = UserIdentity(username="student", git_email="you@university.edu", is_charles=False)
opts = SlurmTemplateOptions(
    sweep_index=5, config_name="optimal", num_sessions=7, num_folds=5,
    nodes=1, ntasks=1, cpus_per_task=4, time="24:00:00", mem="32G",
    output_dir="/scratch/sweeps", array_throttle=10,
    venv_activate="/home/student/.venv/bin/activate",
    repo_dir="/home/student/neural_data_decoding")
script = render_slurm_template(opts, identity=me)
print(script)

That's a complete, submittable `.slurm` script, generated from Python. The load-bearing lines:

* `#SBATCH --array=1-35%10` — expand into 35 tasks (7×5), at most 10 running at once (the *throttle*, to be a good cluster citizen).
* `#SBATCH --time`, `--mem`, `--cpus-per-task` — the resource request per task.
* The body: `cd` to the repo, activate the venv ([00.7](../00_orientation/00.7_pip_packaging_and_project_anatomy.ipynb)), then `python -m neural_data_decoding train --config-name optimal --sweep-index 5 --session-run-idx $SLURM_ARRAY_TASK_ID`.

Every array task runs the *same* command with a *different* `$SLURM_ARRAY_TASK_ID`, so task 1 trains (config 5, its session/fold 1), task 2 the next, and so on. One script, 35 runs, submitted with a single `sbatch`. The header comments record the MATLAB `(SLURMChoice, SLURMIDX)` this index came from — traceability back to the original sweep.

### 2.5 — `sweep-emit-slurm`: the CLI

`render_slurm_template` is wrapped by the `sweep-emit-slurm` subcommand, so you generate the script from the shell:

```bash
python -m neural_data_decoding sweep-emit-slurm --sweep-index 5 --config-name optimal > run.slurm
sbatch run.slurm      # submit the array job to the cluster
```

You emit a `.slurm` per config you want to run (or script a loop over sweep indices), then `sbatch` each. The Python code, the config system ([09.3](09.3_hydra_config_composition.ipynb)), and the sweep table ([09.4](09.4_parameter_sweeps.ipynb)) all feed this one output: a portable, cluster-native job script.

## Section 3 — The `neural_data_decoding` implementation

`render_slurm_template` — the array spec and the option surface:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/sweeps/slurm_template.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "array_spec = " in line)
print(f"{i + 1:4} | {src[i]}")
# and the SlurmTemplateOptions fields:
j0 = next(j for j, line in enumerate(src) if line.startswith("class SlurmTemplateOptions"))
print()
from neural_data_decoding.sweeps.slurm_template import SlurmTemplateOptions
print("SlurmTemplateOptions fields:")
for f in SlurmTemplateOptions.__dataclass_fields__:
    print(f"   {f}")

Things to spot:

* `array_spec = f"1-{total_tasks}%{options.array_throttle}"` — the `--array` value: tasks `1..(sessions×folds)`, `%throttle` capping concurrency. The single line that turns a config into N parallel jobs.
* `SlurmTemplateOptions` is a dataclass carrying everything the script needs — `sweep_index`, `config_name`, `num_sessions`/`num_folds` (the array size), the resource request (`time`, `mem`, `cpus_per_task`), `output_dir`, `array_throttle`, and the environment (`venv_activate`, `repo_dir`).
* `render_slurm_template(options, *, identity=None)` resolves the user identity for the `--mail-user` line — gated so only the primary user gets email (a placeholder identity, or `is_charles=False`, suppresses it). This is the `user_identity.py` module's job.
* `write_slurm_template` (alongside) writes the rendered string to a file; `sweep-emit-slurm` in `cli.py` is the shell entry point.
* The dispatcher's `lookup`/`total_sweep_count`/`iter_by_choice` ([09.4](09.4_parameter_sweeps.ipynb)) provide the config axis; the template provides the session×fold axis — together they span the full sweep.

## Section 4 — Hands-on exercises

### Exercise 4.1 — array size scales with sessions × folds

Show that the array spec's task count is exactly `num_sessions × num_folds`, so more sessions or folds means a bigger array — no code change.

In [ ]:
# Reveal:
for ns, nf in [(7, 5), (10, 5), (7, 10)]:
    opts = SlurmTemplateOptions(sweep_index=1, config_name="optimal", num_sessions=ns, num_folds=nf,
        nodes=1, ntasks=1, cpus_per_task=4, time="24:00:00", mem="32G", output_dir="/s",
        venv_activate="/v", repo_dir="/r")
    line = next(l for l in render_slurm_template(opts, identity=me).splitlines() if "--array" in l)
    print(f"sessions={ns}, folds={nf} → {line.strip()}  ({ns * nf} tasks)")

### Exercise 4.2 — the index is a stable name

Show that `lookup(index)` is deterministic — the same index always names the same config — so a sweep index in a job log unambiguously identifies the run.

In [ ]:
# Reveal:
a = dispatcher.lookup(42)
b = dispatcher.lookup(42)
print(f"index 42 → '{a.description}' with overrides {a.overrides}")
print("same every lookup?", a.description == b.description and a.overrides == b.overrides)
print("→ the integer is a stable, reproducible name for a config — reproducibility by index.")

## Section 5 — Common errors

### The array job runs the wrong (session, fold)

The `$SLURM_ARRAY_TASK_ID` → `(session, fold)` decomposition must match MATLAB's fold-across-sessions ordering (§2.3). A row-major vs fold-major mistake silently trains the wrong combinations. The CLI's decomposition is the single source of truth.

### The whole array runs at once and swamps the cluster

Missing or too-high `array_throttle` (the `%N` in `--array`). Set a sane throttle (§2.4) so you use a fair share, not all of it — clusters have per-user limits and etiquette.

### Jobs fail: "no such file" for the venv or repo

`venv_activate` / `repo_dir` point at *local* paths, not the cluster's (§2.4). These must be the paths on the cluster filesystem ([09.1](09.1_environment_detection.ipynb)), where the job actually runs.

### `sbatch` rejects the script

A resource request the cluster won't grant (`--time` over the partition limit, `--mem` too high) or a malformed `#SBATCH` line. Check the values against the target partition's limits.

### Emitting one script for the whole 147-config sweep

Each `.slurm` is *one* config's session×fold array (§2.4). To run all configs, emit a script per sweep index (or loop `sweep-emit-slurm` over `1..total_sweep_count()`), then `sbatch` each — don't expect one script to cover the config axis too.

## Section 6 — Further reading

- [`src/neural_data_decoding/sweeps/slurm_template.py`](../../src/neural_data_decoding/sweeps/slurm_template.py) — `render_slurm_template`, `SlurmTemplateOptions`.
- [`src/neural_data_decoding/sweeps/dispatcher.py`](../../src/neural_data_decoding/sweeps/dispatcher.py) — the single-integer index (`lookup`, `total_sweep_count`).
- [SLURM job array docs](https://slurm.schedmd.com/job_array.html) — the `--array` mechanism.

Next up: **[09.3 Hydra config composition](09.3_hydra_config_composition.ipynb)** — how `--config-name` and overrides compose the config each job runs.